In [ ]:
# Setup and imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

project_root = Path().absolute()
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import gymnasium as gym
import torch
import subprocess
from datetime import datetime, timedelta

# Import RL environment components
from rl_env import (
    register_cleanrl_env, 
    NewsInterpreter,
    train_uncertainty_forecaster
)

Device: cpu


In [2]:
# Data preparation: Load real historical data with technical indicators and news documents
# NOTE: This cell takes 2-3 minutes to run
from rl_utils.data import prepare_data_features

ticker = 'AAPL'
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365*1.5)).strftime('%Y-%m-%d')  # 1.5 years of data
timeframe = '1D'

# Fetch real data with indicators, news embeddings, and news documents
data, news_documents = prepare_data_features(ticker, start_date, end_date, timeframe)

data.head()

,symbol,Open,High,Low,Close,Volume,EMA_FAST,EMA_MEDIUM,EMA_SLOW,SMA_SHORT,...,CCI,ULTOSC,WILLR,RANGE_HIGH,RANGE_LOW,PULLBACK_REF_HIGH,PULLBACK_REF_LOW,return_1d,return_3d,return_5d
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-08-14 04:00:00+00:00,AAPL,220.57,223.0300,219.7000,221.72,41960574.0,217.551182,217.972696,214.149951,216.612000,...,49.680651,62.815080,-13.108108,222.08,207.23,221.72,207.23,0.002034,0.025342,0.056715
2024-08-15 04:00:00+00:00,AAPL,224.60,225.3500,222.7600,224.72,46414013.0,218.854603,218.586088,214.564463,217.094000,...,94.555771,63.839405,-2.972973,224.72,207.23,224.72,207.23,0.013531,0.033053,0.053490
2024-08-16 04:00:00+00:00,AAPL,223.92,226.8271,223.6501,226.05,44340240.0,220.162857,219.264625,215.014876,217.633333,...,112.198011,67.521473,-2.520834,226.05,207.23,226.05,209.82,0.005918,0.021603,0.045366
2024-08-19 04:00:00+00:00,AAPL,225.72,225.9900,223.0400,225.89,40687813.0,221.204156,219.866932,215.441351,218.143333,...,103.932044,68.104049,-3.039858,226.05,207.23,226.05,213.31,-0.000708,0.018808,0.038431
2024-08-20 04:00:00+00:00,AAPL,225.77,227.1700,225.4500,226.51,30299033.0,222.168855,220.470847,215.875416,218.657333,...,122.385392,66.146410,-2.117421,226.51,207.23,226.51,216.24,0.002745,0.007965,0.023681


In [3]:
# Configure rolling window parameters
TRAIN_WINDOW = 200        # Days for training
TEST_WINDOW = 30          # Days for testing
FORECASTER_LOOKBACK = 60  # LSTM input window
FORECAST_HORIZON = 4      # Days ahead to forecast

# Calculate rolling windows
total_days = len(data)
num_windows = max(1, (total_days - TRAIN_WINDOW) // TEST_WINDOW)

print(f"Total data: {total_days} days")
print(f"Train window: {TRAIN_WINDOW} days")
print(f"Test window: {TEST_WINDOW} days")
print(f"Forecaster lookback: {FORECASTER_LOOKBACK} days")
print(f"Forecast horizon: {FORECAST_HORIZON} days")
print(f"Number of rolling windows: {num_windows}")

# Create window definitions
windows = []
for i in range(num_windows):
    train_start = i * TEST_WINDOW
    train_end = train_start + TRAIN_WINDOW
    test_start = train_end
    test_end = min(test_start + TEST_WINDOW, total_days)
    
    # Need at least 10 days for testing
    if test_end - test_start < 10:
        break
    
    windows.append({
        'window_id': i,
        'train_range': (train_start, train_end),
        'test_range': (test_start, test_end)
    })

Total data: 311 days
Train window: 200 days
Test window: 30 days
Forecaster lookback: 60 days
Forecast horizon: 4 days
Number of rolling windows: 3


In [4]:
# Initialize shared models for incremental learning across windows
global_forecaster = None
global_interpreter = None

# Select feature columns (exclude target and non-numeric)
feature_columns = [c for c in data.columns 
                   if c not in ['Close'] and data[c].dtype in [np.float64, np.float32, np.int64]]

print(f"Using {len(feature_columns)} features for forecaster:")
print(f"  {feature_columns}")

# Storage for results across windows
all_results = []

Using 39 features for forecaster:
  ['Open', 'High', 'Low', 'Volume', 'EMA_FAST', 'EMA_MEDIUM', 'EMA_SLOW', 'SMA_SHORT', 'SMA_MEDIUM', 'SMA_LONG', 'H-L', 'H-PC', 'L-PC', 'TR', 'ATR', '+DM', '-DM', 'TR_DMI', '+DM_DMI', '-DM_DMI', '+DI', '-DI', 'Vol_MA', 'RVOL', 'VOL_EXPAND', 'OBV', 'OBV_MA', 'ROC', 'MFI', 'CCI', 'ULTOSC', 'WILLR', 'RANGE_HIGH', 'RANGE_LOW', 'PULLBACK_REF_HIGH', 'PULLBACK_REF_LOW', 'return_1d', 'return_3d', 'return_5d']


In [ ]:
# Adaptive rolling window training loop using CleanRL scripts

# Import components needed for evaluation
from CleanRL_API.sac_continuous_action import Actor

# Training parameters
TOTAL_TIMESTEPS = 10000  # Reduced for demo (use 50000+ for production)
HIDDEN_SIZE = 256
ALGORITHM = "SAC"  # Choose "SAC" or "PPO"

# Path to CleanRL scripts
sac_script = project_root / "CleanRL_API" / "sac_continuous_action.py"
ppo_script = project_root / "CleanRL_API" / "ppo.py"

# TODO: Loop over each rolling window 
# TODO: figure our how to effectively save and load trained models using cleanRL scripts


WINDOW 0
Train: [0:200] (200 days)
Test:  [200:230] (30 days)

Data: 200 train days, 90 test days (with history)
News: 200 train docs, 90 test docs

[1/5] Training forecaster...

[2/5] Fitting news interpreter...

[2/5] Fitting news interpreter...

[3/5] Registering training environment...

[3/5] Registering training environment...

[4/5] Training agent using SAC...
Training for 10000 timesteps...
Command: /Users/lohitgopikonda/micromamba/envs/cleanrl/bin/python /Users/lohitgopikonda/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask49_Fall2025_CleanRL_Reinforcement_Learning_for_Stock_Trading/CleanRL_API/sac_continuous_action.py --env-id SignalTester-v0-train-w0 --total-timesteps 10000 --seed 42 --hidden-size 256

[4/5] Training agent using SAC...
Training for 10000 timesteps...
Command: /Users/lohitgopikonda/micromamba/envs/cleanrl/bin/python /Users/lohitgopikonda/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask49_Fall2025_CleanRL_Reinforcement_Learning_

FileNotFoundError: Model directory not found for evaluation